# Importing Libraries

In [1]:
import os
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


# Loading The Dataset

In [2]:
labels = os.listdir('/kaggle/input/sanad-dataset') 
# Load dataset 
raw_data = tf.keras.preprocessing.text_dataset_from_directory(
    '../input/sanad-dataset',    
    labels="inferred",           # Inferring labels from the directory structure
    label_mode="int",            # Labels are encoded as integers
    max_length=None,             
    shuffle=True,               
    seed=11,                     
    validation_split=None       
)

# Extracting text and labels from raw_data
x = []  # List to store text data
y = []  # List to store labels


for text_batch, label_batch in raw_data:
    for i in range(len(text_batch)):
        # Convert text from bytes to UTF-8 format and append to 'x'
        s = text_batch.numpy()[i].decode("utf-8") 
        x.append(s)
        
        # Append label names to 'y' by mapping label indices to their corresponding class names
        y.append(raw_data.class_names[label_batch.numpy()[i]])

# Create a DataFrame from the extracted text and labels
data = pd.DataFrame({"text": x, "label": y})


Found 45500 files belonging to 7 classes.


# Data Preprocessing

In [3]:
# Function to clean text data by removing punctuation, diacritics, emojis, and stop words
def clean_text(text):
    """
    Clean text data by removing punctuation, diacritics, emojis, and stop words.

    Args:
    text (str): Input text to be cleaned.

    Returns:
    str: Cleaned text after removing punctuation, diacritics, emojis, and stop words.
    """
    text = "".join([word for word in text if word not in string.punctuation])
    text = remove_emoji(text)
    text = remove_diacritics(text)
    tokens = word_tokenize(text)
    text = ' '.join([word for word in tokens if word not in stop_words])
    return text

# Clean text data
stop_words = list(set(stopwords.words('arabic')))

# Regular expression pattern to remove Arabic diacritics
arabic_diacritics = re.compile("""
                             ّ    | # Tashdid
                             َ    | # Fatha
                             ً    | # Tanwin Fath
                             ُ    | # Damma
                             ٌ    | # Tanwin Damm
                             ِ    | # Kasra
                             ٍ    | # Tanwin Kasr
                             ْ    | # Sukun
                             ـ     # Tatwil/Kashida
                         """, re.VERBOSE)

def remove_diacritics(text):
    """
    Remove Arabic diacritics from the input text.

    Args:
    text (str): Input text containing Arabic diacritics.

    Returns:
    str: Text with Arabic diacritics removed.
    """
    text = re.sub(arabic_diacritics, '', text)
    return text

def remove_emoji(text):
    """
    Remove emojis from the input text.

    Args:
    text (str): Input text containing emojis.

    Returns:
    str: Text with emojis removed.
    """
    regrex_pattern = re.compile(pattern="["
                                  u"\U0001F600-\U0001F64F"  # emoticons
                                  u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                                  u"\U0001F680-\U0001F6FF"  # transport & map symbols
                                  u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                                  "]+", flags=re.UNICODE)
    return regrex_pattern.sub(r'', text)

# Apply text cleaning functions to the 'text' column
data['cleaned_text'] = data['text'].apply(clean_text)

In [4]:
# Tokenization
max_words = 10000  
max_sequence_length = 100  

# Initialize a Tokenizer with a maximum number of words
tokenizer = Tokenizer(num_words=max_words)

# Fit the Tokenizer on the cleaned text data to generate word indices
tokenizer.fit_on_texts(data['cleaned_text'])

# Convert text sequences to numerical sequences using the fitted Tokenizer
sequences = tokenizer.texts_to_sequences(data['cleaned_text'])

# Retrieve the word index from the Tokenizer
word_index = tokenizer.word_index

# Pad sequences to ensure uniform length for modeling
X = pad_sequences(sequences, maxlen=max_sequence_length)

# Extract labels from the data
y = data['label']

# Convert categorical labels to numerical form using LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


# Building RNN Model

In [5]:
embedding_dim = 100

model = Sequential()

# Add an Embedding layer to the model
model.add(Embedding(max_words, embedding_dim, input_length=max_sequence_length))

# Add a Bidirectional LSTM layer to the model
model.add(Bidirectional(LSTM(128)))

# Add a Dense output layer with a softmax activation function
# The number of units in the Dense layer is set to the number of unique labels
model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

# Compile the model 
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model using the entire dataset
model.fit(X, y_encoded, epochs=10, batch_size=32)


Epoch 1/10
1422/1422 [==============================] - 95s 61ms/step - loss: 0.4772 - accuracy: 0.8512
Epoch 2/10
1422/1422 [==============================] - 23s 16ms/step - loss: 0.1604 - accuracy: 0.9548
Epoch 3/10
1422/1422 [==============================] - 20s 14ms/step - loss: 0.1174 - accuracy: 0.9667
Epoch 4/10
1422/1422 [==============================] - 17s 12ms/step - loss: 0.0854 - accuracy: 0.9761
Epoch 5/10
1422/1422 [==============================] - 16s 12ms/step - loss: 0.0507 - accuracy: 0.9865
Epoch 6/10
1422/1422 [==============================] - 16s 11ms/step - loss: 0.0439 - accuracy: 0.9882
Epoch 7/10
1422/1422 [==============================] - 15s 11ms/step - loss: 0.0276 - accuracy: 0.9927
Epoch 8/10
1422/1422 [==============================] - 15s 10ms/step - loss: 0.0158 - accuracy: 0.9959
Epoch 9/10
1422/1422 [==============================] - 15s 10ms/step - loss: 0.0125 - accuracy: 0.9965
Epoch 10/10
1422/1422 [==============================] - 15s 11m

# Example

In [6]:
# Create a mapping dictionary for numerical labels to Arabic categories
label_mapping = {
    0: 'الثقافة',    
    1: 'الاقتصاد',
    2: 'الطب',
    3: 'السياسة',
    4: 'الدين',
    5: 'الرياضة',
    6: 'التكنولوجيا'
}

# Function to predict the Category of news based on provided text
def predict_Category(news_text):
    """
    Predict the Category of a given news text.

    Args:
    news_text (str): Input news text to predict the Category.

    Returns:
    str: Predicted Category in Arabic.
    """
    # Clean the user input news text
    cleaned_text = clean_text(news_text)
    
    # Convert the cleaned text into sequences and pad it
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded_sequence = pad_sequences(sequence, maxlen=max_sequence_length)
    
    # Get the prediction from the trained model
    prediction = model.predict(padded_sequence)
    predicted_class = np.argmax(prediction)
    
    # Map the predicted class to the corresponding category
    predicted_Category = label_mapping.get(predicted_class)
    return predicted_Category

# User input to enter news text
user_news = input("Enter your news: ")

# Predict the Category of the provided news text
predicted_Category = predict_Category(user_news)
print("Predicted Category of the news :", predicted_Category)

Enter your news:  منتخب مصر يفوز بكاس العالم


1/1 [==============================] - 1s 684ms/step
Predicted Category of the news : الرياضة
